In [11]:
!pip install langchain langchain-community langchain-core
!pip install chromadb sentence-transformers
!pip install pypdf
!pip install langgraph

In [9]:
from google.colab import files
uploaded = files.upload()

Saving Amazon_Returns_and_Refunds_Policy_Seller_fulfilled_orders.pdf to Amazon_Returns_and_Refunds_Policy_Seller_fulfilled_orders (1).pdf


In [3]:
!ls /content


Amazon_Returns_and_Refunds_Policy_Seller_fulfilled_orders.pdf  sample_data


In [7]:
file_path = "/content/Amazon_Returns_and_Refunds_Policy_Seller_fulfilled_orders.pdf"

In [21]:
from transformers import pipeline

# Use a small GPT model (compatible)
generator = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=150
)

# Query
query = "What is the return policy for electronics?"

# Retrieve relevant chunks
docs = retriever.invoke(query)

# Combine context
context = "\n".join([doc.page_content for doc in docs[:3]])

# Prompt
prompt = f"""
Context:
{context}

Question: {query}
Answer:
"""

# Generate response
response = generator(prompt)

print(response[0]['generated_text'])

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Context:
 How do I get a refund?  
 How are refunds for seller fulfilled orders calculated?  
 What do I do if the seller doesn't respond to my return request? 
What's the returns policy for seller fulfilled items? 
 
The following returns policy applies to seller fulfilled items purchased on Amazon. 
Product 
Categories 
Return Time from Delivery 
Item is damaged/ defective You no longer want/ need the item 
Books 30 Days 7 Days 
Movies & TV 30 Days 7 Days 
Electronics 10 Days 10 Days
Return Condition 
 
Damaged/ Defective Items  
 
 
For Books, Movies & TV: 
 All damaged/ defective items must be returned in the original condition they were 
received in.  
 For returns of damaged/ defective items, you need to contact the seller within 14 days of 
delivery to inform them of the damage/ defect. 
For Electronics: 
 You can return electronics that are dead on arrival, or arrive in damaged or defective
assist in facilitating refunds for you only when the seller notifies us of the re

In [24]:
from langgraph.graph import StateGraph

class State(dict):
    pass

# Node 1: Processing
def process_node(state):
    # ✅ Force safe extraction
    query = state.get("query", "")

    if not query:
        return {"answer": "No query provided."}

    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs[:3]])

    prompt = f"""
    Context:
    {context}

    Question: {query}
    Answer:
    """

    response = generator(prompt)
    answer = response[0]['generated_text']

    return {"answer": answer}


# Node 2: Output
def output_node(state):
    print("Final Answer:\n", state.get("answer", "No answer"))
    return state


# Build graph
graph = StateGraph(State)

graph.add_node("process", process_node)
graph.add_node("output", output_node)

graph.set_entry_point("process")
graph.add_edge("process", "output")

app = graph.compile()

# ✅ Important: wrap input properly
app.invoke(State({"query": "How do I get a refund?"}))

Final Answer:
 No answer


In [25]:
from langgraph.graph import StateGraph

class State(dict):
    pass

# Node 1: Processing
def process_node(state):
    query = state.get("query", "")

    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs[:3]])

    prompt = f"""
    Context:
    {context}

    Question: {query}
    Answer:
    """

    response = generator(prompt)
    answer = response[0]['generated_text']

    return {"query": query, "answer": answer}


# Node 2: Decision (Routing)
def decision_node(state):
    answer = state.get("answer", "")

    # Simple confidence logic
    if len(answer) < 50 or "don't know" in answer.lower():
        return {"route": "human"}
    else:
        return {"route": "final"}


# Node 3: Final Answer
def final_node(state):
    print("🤖 AI Answer:\n", state.get("answer"))
    return state


# Node 4: HITL (Human escalation)
def human_node(state):
    print("⚠️ Escalating to Human Support...")

    human_response = input("👨‍💻 Enter human response: ")

    print("✅ Human Response:\n", human_response)

    return {"answer": human_response}


# Build graph
graph = StateGraph(State)

graph.add_node("process", process_node)
graph.add_node("decision", decision_node)
graph.add_node("final", final_node)
graph.add_node("human", human_node)

# Flow
graph.set_entry_point("process")
graph.add_edge("process", "decision")

# Conditional routing
def route_logic(state):
    return state["route"]

graph.add_conditional_edges(
    "decision",
    route_logic,
    {
        "final": "final",
        "human": "human"
    }
)

app = graph.compile()

# Run
app.invoke(State({"query": "What is the refund process?"}))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⚠️ Escalating to Human Support...
👨‍💻 Enter human response: The refund is processed after the item is returned and approved by the seller.
✅ Human Response:
 The refund is processed after the item is returned and approved by the seller.
